[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_sand_shape_similarity.ipynb)

# Shape similarity to silymarin

**Blue group · Cryptosporidiosis**

Two molecules can be built from different parts and still fill the same space, which is often what
decides whether they bind the same pocket. This notebook asks which of the 28,732 purchasable hits
resemble silymarin in **shape**, and whether that picks out different molecules from the usual
comparison of chemical structure.

## What you will do

- Load the shape descriptors of the hits and of silymarin
- Measure how close each hit is to silymarin, and see the whole distribution
- Compare that with the similarity between random pairs of hits, to know what is high
- Do the same with a classic 2D fingerprint similarity, and see how far the two agree
- List and draw the closest molecules, and save the ranked table

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Shape, and how we get it

A pocket is a hole of a particular size and form. A molecule that fills it well can bind even if it
is built from quite different pieces than the molecule you started from. Chemists call this
**scaffold hopping**, and it is one of the main ways to move away from a starting compound while
keeping the activity.

Comparing shapes properly is slow: you have to work out the three-dimensional forms each molecule
can fold into, then try to lay one over the other. **SAND**, the Ersilia model
[eos5mnx](https://github.com/ersilia-os/eos5mnx), skips that. It reads the flat structure and
returns **512 numbers** that describe the shape the molecule would take. Comparing two molecules is
then a single arithmetic step.

The model was trained to agree with a real 3D overlay program, and reaches a correlation of about
0.86 with it. That is good, but it is a prediction of an overlay score, not the score itself.

SAND's authors never say what counts as "shape-similar". They never draw a line and call anything
above it a hit. Instead they check whether the molecules SAND puts at the **top of the list** are the
ones a real 3D overlay would also put there, and how far down the list you have to go to find them.
In their own plots, the best matches of a molecule reach a cosine of about 0.6, and most sit well
below that.

So a score of 0.3 means nothing on its own. What it means depends on the company it keeps, which is
what section 4 is for.

## 2. Load the descriptors

Two files, both from `eos5mnx`. The hits file is 183 MB, too large for the repository, so it lives
in the group's Drive folder under **Projects/BlueTeam/Data**. Download it from there first: in
Colab the cell asks you to upload it, and locally it is read from `data/downloads/`.

The seed file, silymarin on its own, is small enough to sit in `data/`.

> **Note:** the upload takes a few minutes and Colab shows no progress bar until it finishes.

In [ ]:
import sys
from pathlib import Path
from scripts import shape

DOWNLOADS = Path("data/downloads")
DOWNLOADS.mkdir(parents=True, exist_ok=True)
HITS = DOWNLOADS / "eos5mnx_pharmit_hits.csv"

if "google.colab" in sys.modules and not HITS.exists():
    from google.colab import files
    print("Upload eos5mnx_pharmit_hits.csv from Drive")
    for name in files.upload():
        Path(name).rename(HITS)

print(f"{HITS.name}: {HITS.stat().st_size / 1e6:.0f} MB")

Reading the file takes up to a minute. Each molecule becomes one row of 512 numbers.

In [ ]:
smiles, vectors = shape.load_embeddings(HITS)
seed_smiles, seed_vectors = shape.load_embeddings("data/eos5mnx_silymarin.csv")
seed = seed_vectors[0]

print(f"{len(smiles)} hits, {vectors.shape[1]} numbers each")
print(f"seed: {seed_smiles[0]}")

The descriptors are **normalised**: every molecule's 512 numbers, taken as a point in space,
sit exactly one unit from the origin. That is what lets us compare two molecules by multiplying
their numbers together and adding up, a **cosine similarity**. It runs from 1 for the same
direction, through 0, to −1 for opposite directions.

In [ ]:
lengths = shape.check_normalised(vectors)
print(f"vector lengths: min {lengths.min():.3f}, max {lengths.max():.3f}")
print(f"silymarin against itself: {float(seed @ seed):.3f}")

## 3. How close are the hits to silymarin?

One multiplication gives every hit's shape similarity to silymarin at once.

In [ ]:
import numpy as np
import pandas as pd

hits = pd.DataFrame({"smiles": smiles})
hits["shape"] = shape.shape_similarity(vectors, seed)

hits["shape"].describe().round(3).to_frame().T

Those numbers are all small, and the largest is nowhere near 1. Before reading anything into
that, look at the shape of the distribution.

In [ ]:
import stylia

figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
ax.hist(hits["shape"], bins=60, color=stylia.NamedColors().cobalt)
stylia.label(ax, xlabel="Shape similarity to silymarin", ylabel="Number of hits")
figure.tight_layout()

## 4. Is that high?

A similarity only means something next to the similarities you would get anyway. So we take a
random sample of the hits and measure every pair within it. That is the **background**: what two
unrelated molecules from this same set score against each other.

If silymarin's neighbours stand out, they should sit to the right of that background.

In [ ]:
background = shape.shape_background(vectors, sample=4000)

comparison = pd.DataFrame({
    "hits vs silymarin": hits["shape"].describe(),
    "random pairs of hits": pd.Series(background).describe(),
}).round(3)
comparison.loc[["mean", "std", "min", "max"]]

Drawn together, with the closest hit marked.

In [ ]:
figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
shape.plot_distribution(ax, hits["shape"].values, background,
                        xlabel="Shape similarity", title="Shape: silymarin vs the background")
figure.tight_layout()

The two distributions sit almost on top of each other, and the hits are if anything slightly
**more** similar to one another than to silymarin. Putting the best hit into the background tells
us how unusual it really is.

In [ ]:
best = hits["shape"].max()
print(f"closest hit: {best:.3f}")
print(f"that beats {shape.percentile_of(best, background):.1f}% of random pairs of hits")

So the closest hit beats about 99 per cent of random pairs. It is not an ordinary molecule from this
set, and it sits inside the range where the SAND paper's own best matches fall.

But look at the whole scale. The background reaches 0.97, because some pairs of hits really do share
a shape, and nothing comes close to that against silymarin. Read the two together: there are
molecules here worth looking at, and none of them is a shape copy of the seed.

**The background is not a neutral one, and that matters.** These are not random molecules. Every one
of them was kept because it matched the same six-point pharmacophore, so they all carry the same
features in roughly the same places. Resembling each other is what being in this set means. Silymarin
was never held to that test: the pharmacophore came from the *pocket*, not from silymarin, and it
even asks for an acid group that silymarin does not have. So the hits being closer to each other than
to silymarin is the expected outcome, not a surprise.

That is a real result, not a failure. The hits came from a **pharmacophore** search, which asks for
a handful of features in the right places and says nothing about overall shape. Finding that they
are not silymarin-shaped means the search did what it was meant to do: it went beyond the starting
molecule.

## 5. The other kind of similarity

The usual way to compare molecules is not by shape but by **structure**: list the small fragments
each one contains and count how many they share. That is a **Morgan fingerprint**, and the score is
the **Tanimoto similarity**, from 0 to 1.

It answers a different question. Two molecules can share many fragments and fold differently, or
share almost none and still fill the same space. Let us measure it and compare.

In [ ]:
fingerprints = shape.fingerprints(hits["smiles"])
seed_fingerprint = shape.fingerprints([seed_smiles[0]])[0]

hits["tanimoto"] = shape.tanimoto_similarity(fingerprints, seed_fingerprint)
hits["tanimoto"].describe().round(3).to_frame().T

The same background treatment, on the 2D side.

In [ ]:
tanimoto_background = shape.tanimoto_background(fingerprints, sample=2000)

figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
shape.plot_distribution(ax, hits["tanimoto"].values, tanimoto_background,
                        xlabel="Tanimoto similarity", title="2D: silymarin vs the background")
figure.tight_layout()

The 2D picture tells the same story, and for the same reason: these molecules were chosen for
matching a few features, not for looking like silymarin.

> **Note:** the background stops just short of 0.7 because the hit list was built that way. The
> notebook before this one removed near-copies at Tanimoto 0.7, so no pair above that survives.

## 6. Do the two measures agree?

Each hit now has two scores. If shape and structure said the same thing, the points below would lie
on a line.

In [ ]:
figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
shape.plot_agreement(ax, hits["shape"].values, hits["tanimoto"].values)
figure.tight_layout()

print(f"correlation: {hits['shape'].corr(hits['tanimoto'], method='spearman'):.2f} (Spearman)")

They agree only loosely. The clearest way to see what that costs you is to take the twenty
closest molecules by each measure and count how many appear in both lists.

In [ ]:
by_shape = set(hits.nlargest(20, "shape").index)
by_tanimoto = set(hits.nlargest(20, "tanimoto").index)

print(f"molecules in both top-20 lists: {len(by_shape & by_tanimoto)}")

Almost none. Which measure you use decides which molecules you order, so it is worth saying out loud
which question you are asking: *does it look like silymarin on paper*, or *does it fill the same
space*.

The SAND authors make the same point. In their paper the molecules a fingerprint ranks highest often
turn out to have quite different 3D shapes: the same small pieces, put together differently. That is
the gap you are looking at in the plot above.

## 7. The closest molecules by shape

Bring back the MolPort catalogue numbers, so each molecule can be ordered.

In [ ]:
catalogue = pd.read_csv("data/pharmit_hits_molport.csv")
hits = hits.merge(catalogue, on="smiles", how="left")

neighbours = hits.nlargest(12, "shape").reset_index(drop=True)
neighbours[["molport_id", "shape", "tanimoto"]].round(3)

And the molecules themselves, with silymarin first for comparison. Look at whether the shapes
strike you as similar: the model is predicting how they would fill space, which a flat drawing only
hints at.

In [ ]:
shape.draw_molecules(
    [seed_smiles[0]] + neighbours["smiles"].tolist(),
    ["silymarin (seed)"] + [f"{row.molport_id}\nshape {row.shape:.2f}"
                            for row in neighbours.itertuples()])

> **Exercise:** the cell above takes the 12 closest by shape. Change `nlargest(12, "shape")` to
> sort by `tanimoto` instead and draw those. Do they look more like silymarin to you? Which set
> would you rather order, and why?

## 8. Lay them over silymarin in 3D

Everything so far has been SAND's *prediction* of how well two molecules would overlap. It never
builds a molecule in 3D. Now we do the real thing for a handful of the closest hits, and see whether
the prediction holds.

For each one: work out the shapes it can fold into, lay each of them over silymarin, keep the best
fit, and score it with a **shape Tanimoto**, the fraction of space the two molecules share. That runs
from 0 to 1 on its own scale, which is not SAND's scale.

Silymarin is taken from `data/silymarin_ligand.sdf`, the pose docked in the CpABC1 pocket, so we are
comparing against the form it actually binds in.

> **Note:** this is the slow part, a couple of seconds per molecule. It is exactly the cost SAND
> exists to avoid: doing it for all 28,732 hits would take hours.

In [ ]:
from rdkit import Chem

reference = Chem.MolFromMolFile("data/silymarin_ligand.sdf")
closest = neighbours.head(10)

overlays = []
for row in closest.itertuples():
    molecule, conformer, overlap = shape.overlay_on(row.smiles, reference)
    overlays.append({"molport_id": row.molport_id, "sand": row.shape,
                     "overlay": overlap, "molecule": molecule, "conformer": conformer})

pd.DataFrame(overlays)[["molport_id", "sand", "overlay"]].round(3)

Now look at them. Silymarin is grey, the hit is blue. Drag to rotate.

What you are looking for is whether the blue molecule fills the same space as the grey one, not
whether the two are built the same way. A good shape match can look nothing like the seed on paper.

In [ ]:
for entry in overlays[:3]:
    print(f"{entry['molport_id']}  SAND {entry['sand']:.2f}  overlay {entry['overlay']:.2f}")
    shape.view_overlay(reference, entry["molecule"], entry["conformer"]).show()

Those numbers mean little without something to compare them with, so take the same measurement
on molecules SAND did **not** pick: a random handful from the same hit list.

In [ ]:
import numpy as np

generator = np.random.default_rng(0)
control = hits.iloc[generator.choice(len(hits), 10, replace=False)]

control_overlaps = [shape.overlay_on(row.smiles, reference)[2] for row in control.itertuples()]

print(f"SAND's closest 10: overlay {np.mean([e['overlay'] for e in overlays]):.3f}")
print(f"10 random hits:    overlay {np.mean(control_overlaps):.3f}")

The molecules SAND ranks highest do overlay silymarin better than random hits do, so the ranking
is telling us something real. But the gap is small, and for the reason in section 4: a random hit
here is not a random molecule. It already matches the same pharmacophore, so it already has a fair
chance of filling a similar space. The honest reading is that SAND is sorting sensibly inside a set
that was already narrow.

> **Exercise:** raise the number of molecules in both groups and see whether the gap holds up. Ten
> against ten is a small test, and the two averages are close enough that it could go either way.

## 9. Save the ranked list

The whole table goes to `outputs/`, sorted by shape similarity, with both scores and the catalogue
number. Upload it to **Projects/BlueTeam/Data** so the group can work from it.

In [ ]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

ranked = hits.sort_values("shape", ascending=False)[["smiles", "molport_id", "shape", "tanimoto"]]
ranked.to_csv(OUT / "sand_shape_similarity_to_silymarin.csv", index=False)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(OUT / "sand_shape_similarity_to_silymarin.csv"))
ranked.head()

## Summary

- Measured the shape similarity of all 28,732 hits to silymarin with SAND, a model that predicts 3D
  shape from the flat structure.
- Against a background of random pairs of hits, none of them is a shape copy of silymarin: the
  closest scores about as high as an ordinary well-matched pair. The pharmacophore search moved away
  from the starting molecule, which is what it was for.
- Shape similarity and 2D fingerprint similarity agree only loosely, and their top-20 lists barely
  overlap, so the choice of measure decides which molecules you order.

**Next:** decide which measure matters for this project, then narrow the ranked list with the
models in the Ersilia Model Hub, as `blue_chemical_space.ipynb` starts to do.